In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [3]:
from multi_task_training.rnn_model import PLRNN, train_multitask

## define tasks

In [4]:
from dataset import MultiTaskDataset, collate_fn
from tasks import DelayedResponse, CategoryDecision ,DelayedMatchToSample, GoNogo, MemoryArithmetics, Arithmetics, ContextIntegration, PredictiveTracking, NoiseCleaner, CopyTask, MotorPerturbation, DelayedResponse32D, DelayComparison, DelayMatchCategory, DelayedMatchToSample, DualDelayMatchSample, DurationEstimation, IntervalDiscrimination, ToneDetection
from torch.utils.data import Dataset, DataLoader
# Set random seeds
torch.manual_seed(0)
np.random.seed(0)
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [10]:

# Duration parameters (in timesteps)
#duration_params = {
#    'context': (5, 15),
#    'stimulus': (10, 30),
#    'delay': (10, 30),
#    'response': (5, 15)
#}

from delayed_response import PulseDecisionMaking


duration_params = {
    'context': (5, 10),      # Shorter
    'stimulus': (10, 15),    # Shorter, less variable
    'delay': (10, 15),       # Shorter delays
    'response': (5, 10)
}

duration_params_hard = {
    'fixation': (10, 20),     # Longer, more variable
    'stimulus': (10, 40),    # More variable stimulus
    'delay': (20, 50),       # Longer memory requirement
    'response': (10, 20)
}

# Create tasks
tasks = [
    # # DelayedResponse32D(duration_params),
    # DelayComparison(duration_params, min_diff=0.15),
    # DelayMatchCategory(duration_params),
    # DelayedMatchToSample(duration_params, mode='match'),
    # DelayedMatchToSample(duration_params, mode='nonmatch'),
    # DualDelayMatchSample(duration_params),
    # GoNogo(duration_params),
    # DurationEstimation(duration_params),
    # IntervalDiscrimination(duration_params),
    # # MultiSensoryIntegration(duration_params),
    # ToneDetection(duration_params),
    # # PulseDecisionMaking(duration_params),
    # # NextPositionPrediction(duration_params),
    # PredictiveTracking(duration_params),
    # # NoiseCleaner(duration_params, noise_level=0.5),
    # Arithmetics(duration_params, mode='multiply'),
    # Arithmetics(duration_params, mode='add'),
    # # MemoryArithmetics(duration_params, mode="substact"),
    # # DelayedResponse(duration_params, mode='pro'),
    # # DelayedResponse(duration_params, mode='anti'),
    # # CategoryDecision(duration_params, mode='pro'),
    # # CategoryDecision(duration_params, mode='anti'),
    # # DelayedMatchToSample(duration_params, mode='match'),
    # # DelayedMatchToSample(duration_params, mode='nonmatch'),
    # # ContextIntegration(duration_params, relevant_modality=1), 
    # # ContextIntegration(duration_params, relevant_modality=2),
    # # MotorPerturbation(duration_params),
    CopyTask(duration_params, seq_len=8, num_symbols=4, delay_range=0)
]

task_names = []

print(f"Training on {len(tasks)} tasks: {', '.join(task_names)}")

# Create dataset and dataloader
train_dataset = MultiTaskDataset(tasks, n_trials=500*len(tasks))
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    shuffle=True, 
    collate_fn=collate_fn
)

n_test_trials=999

test_dataset = MultiTaskDataset(tasks, n_trials=n_test_trials)
test_loader = DataLoader(
    test_dataset,
    batch_size=2000,
    shuffle=False,
    collate_fn=collate_fn
)


Training on 1 tasks: 


In [11]:
# Get one batch
inputs, targets, masks, task_ids = next(iter(train_loader))

In [12]:
print(inputs.shape)

torch.Size([16, 25, 34])


## define model

In [13]:
# Model parameters
M = 64  # Total hidden units
L = 4  # Number of latent units (with ReLU)
N = 33   # Output dimension (must match task OUTPUT_DIM)
input_dim = inputs.shape[-1]  # stimulus + task identity (n_tasks)

# Create model
model = PLRNN(M, L, N, input_dim).to(device)
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [14]:
# Train
model.train()
loss_history, task_accuracies, _, _, _, _, _, _ = train_multitask(
    model, train_loader, optimizer, device, tau=0.01, M_reg=int(model.M/2), num_epochs=200,
)

Epoch 5/200, Loss: 0.6684
  Task 0 Train Acc: 20.82%
Epoch 10/200, Loss: 0.4893
  Task 0 Train Acc: 28.78%
Epoch 15/200, Loss: 0.3602
  Task 0 Train Acc: 37.77%
Epoch 20/200, Loss: 0.2799
  Task 0 Train Acc: 42.58%
Epoch 25/200, Loss: 0.2076
  Task 0 Train Acc: 49.78%
Epoch 30/200, Loss: 0.1431
  Task 0 Train Acc: 62.00%
Epoch 35/200, Loss: 0.0892
  Task 0 Train Acc: 78.67%
Epoch 40/200, Loss: 0.0747
  Task 0 Train Acc: 81.50%
Epoch 45/200, Loss: 0.0460
  Task 0 Train Acc: 90.05%
Epoch 50/200, Loss: 0.0362
  Task 0 Train Acc: 92.83%
Epoch 55/200, Loss: 0.0274
  Task 0 Train Acc: 95.00%
Epoch 60/200, Loss: 0.0031
  Task 0 Train Acc: 100.00%
Epoch 65/200, Loss: 0.0017
  Task 0 Train Acc: 100.00%
Epoch 70/200, Loss: 0.0011
  Task 0 Train Acc: 100.00%
Epoch 75/200, Loss: 0.0007
  Task 0 Train Acc: 100.00%
Epoch 80/200, Loss: 0.0006
  Task 0 Train Acc: 100.00%
Epoch 85/200, Loss: 0.0005
  Task 0 Train Acc: 100.00%
Epoch 90/200, Loss: 0.0005
  Task 0 Train Acc: 100.00%
Epoch 95/200, Loss: 0.

ValueError: too many values to unpack (expected 8)

In [36]:
from pathlib import Path
def save_model(model, optimizer, filepath): #loss_history, task_accuracies, 
    """Save model checkpoint"""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
      #  'loss_history': loss_history,
       # 'task_accuracies': task_accuracies,
        'model_config': {
            'M': model.M,
            'L': model.L,
            'N': model.N,
            'input_dim': model.input_dim
        }
    }, filepath)

def load_model(filepath, model=None, optimizer=None, device='cpu'):
    """Load model checkpoint"""
    checkpoint = torch.load(filepath, map_location=device)
    cfg = checkpoint['model_config']
    model = PLRNN(cfg['M'], cfg['L'], cfg['N'], cfg['input_dim']).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    return model, optimizer#, checkpoint['loss_history'], checkpoint['task_accuracies']

In [37]:
# Save
save_model(model, optimizer, 'DualDelayMatchSample_Solver.pt') #loss_history, task_accuracies, 

In [ ]:
# Load
#model, optimizer = load_model('checkpoints/11_tasks_model_M64_L4_2.pt', device=device)

In [15]:
n_test_trials=90000
# Create test dataset
test_dataset = MultiTaskDataset(tasks, n_trials=n_test_trials)
test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False,
    collate_fn=collate_fn
)

In [16]:
from multi_task_training.rnn_model import evaluate_model

In [17]:
task_names = [type(t).__name__ for t in tasks]

In [18]:
# Get test accuracies
print("Evaluating test accuracy...")
test_accuracies = evaluate_model(model, test_loader, device, task_names, tasks=tasks)

print("\nTest Accuracies:")
for task_id, acc in test_accuracies.items():
    print(f"  {task_names[task_id]}: {acc:.2%}")

# latent_states, trial_info = get_latent_states(model, test_loader, device, task_names)

Evaluating test accuracy...

Test Accuracies:
  CopyTask: 100.00%


In [ ]:
latent_states[0][0].shape

In [ ]:
import analysis_functions as af
import importlib
importlib.reload(af)
import numpy as np

In [ ]:
# Compute distributions for all tasks
distributions = {}
for task_id in range(len(task_names)):
    if latent_states[task_id].shape[0] > 0:
        distributions[task_id] = af.bitcode_distribution(latent_states[task_id], model.L)

# Align distributions
aligned_matrix, bitcode_labels = af.align_bitcode_distributions(distributions)


## Plotting

In [ ]:
plt.rcParams['font.size'] = 18
plt.rcParams['lines.linewidth'] = 2.
# Usage:
af.plot_bitcode_distributions(latent_states, model, task_names, aligned_matrix, bitcode_labels, top_k=16)

In [ ]:
plt.rcParams['font.size'] = 12
# Plot individual task distributions
fig, axes = plt.subplots(3, 4, figsize=(15, 8))
axes = axes.flatten()

top_k=20

for task_id, ax in enumerate(axes):
    if task_id < len(task_names) and latent_states[task_id].shape[0] > 0:
        dist = distributions[task_id]
        # Get top bitcodes for this task
        sorted_items = sorted(dist.items(), key=lambda x: x[1], reverse=True)[:top_k]
        bitcodes, freqs = zip(*sorted_items) if sorted_items else ([], [])
        
        ax.bar(range(len(bitcodes)), freqs)
        ax.set_xticks(range(len(bitcodes)))
        ax.set_xticks([])
        #ax.set_xticklabels(bitcodes, rotation=90, fontsize=6)
        ax.set_xlabel('Bitcode')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{task_names[task_id]} (n={len(dist)} bitcodes)')

plt.tight_layout()
#plt.savefig('bitcode_distributions_individual.png', dpi=150, bbox_inches='tight')
plt.show()

## analyze similarity between task reps

In [ ]:
# Assuming your matrix is (11, 8)
def compute_similarity_matrix(prob_matrix, metric='js'):
    n = prob_matrix.shape[0]
    sim_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            if metric == 'js':
                # Jensen-Shannon distance (scipy returns sqrt of JS divergence)
                dist = jensenshannon(prob_matrix[i], prob_matrix[j])
            elif metric == 'hellinger':
                dist = np.sqrt(1 - np.sum(np.sqrt(prob_matrix[i] * prob_matrix[j])))
            elif metric == 'bhattacharyya':
                dist = -np.log(np.sum(np.sqrt(prob_matrix[i] * prob_matrix[j])))
            sim_matrix[i, j] = sim_matrix[j, i] = dist
    
    return sim_matrix

In [ ]:
from scipy.spatial.distance import jensenshannon
# Or convert distance to similarity
sim_matrix_3_2 = 1 - compute_similarity_matrix(aligned_matrix, 'js')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# First matrix
im1 = axes[0].imshow(sim_matrix_3_2, cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Similarity Matrix P=5/64, 1st run")
axes[0].set_xlabel("Task")
axes[0].set_ylabel("Task")
plt.colorbar(im1, ax=axes[0])

# Second matrix
im2 = axes[1].imshow(sim_matrix_5_3, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Similarity Matrix P=5/64, 2nd run")
axes[1].set_xlabel("Task")
axes[1].set_ylabel("Task")
plt.colorbar(im2, ax=axes[1])

# Absolute difference
diff = np.abs(sim_matrix_5_2 - sim_matrix_5_3)
im3 = axes[2].imshow(diff, cmap="Blues", vmin=0, vmax=1)
#axes[2].set_title("Similarity Matrix P=5/64")
axes[2].set_title("Absolute Difference")#="+str(np.round(np.median(diff),2)))
axes[2].set_xlabel("Task")
axes[2].set_ylabel("Task")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
corr=np.corrcoef(sim_matrix_3_2.flatten(), sim_matrix_5_3.flatten())[0,1]
print(corr)

In [ ]:
from collections import defaultdict

def compute_global_distribution(distributions):
    # Collect all unique bitcodes
    all_bitcodes = set()
    for task_dist in distributions.values():
        all_bitcodes.update(task_dist.keys())
    
    # Compute average probability for each bitcode
    global_dist = {}
    n_tasks = len(distributions)
    
    for bitcode in all_bitcodes:
        probs = [distributions[task].get(bitcode, 0.0) for task in distributions.keys()]
        global_dist[bitcode] = np.mean(probs)
    
    return global_dist


def plot_global_distribution(global_dist):
    """Plot the global distribution as a bar chart."""
    # Sort by usage
    sorted_regions = sorted(global_dist.items(), key=lambda x: x[1], reverse=True)
    
    bitcodes = [b for b, _ in sorted_regions]
    probs = [p for _, p in sorted_regions]
    
    plt.figure(figsize=(12, 6))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(bitcodes)))
    plt.bar(range(len(bitcodes)), probs, color=colors, edgecolor='black', linewidth=0.5)
    plt.xlabel('Subregion')
    plt.ylabel('Average Probability Across Tasks')
  #  plt.title('Global Subregion Usage Distribution', fontsize=14, fontweight='bold')
    plt.xticks(range(len(bitcodes)), bitcodes, rotation=45, ha='right')

    plt.tight_layout()
    
    print(f"Total unique subregions: {len(global_dist)}")
    print(f"Top 5 regions: {sorted_regions[:5]}")
    
    return plt.gcf()




In [ ]:
plt.rcParams['font.size'] = 16
global_dist = compute_global_distribution(distributions)
fig = plot_global_distribution(global_dist)
plt.show()